In [ ]:
from IPython.display import Image, display, Math

#funzione per plottare in LaTex delle matrici
def array_to_latex(array, real = False, array_name = None):
    array = array.real if real else array
    matrix = ''
    for row in array:
        try:
            for number in row:
                matrix += f'{number}&'
        except TypeError:
            matrix += f'{row}&'
        matrix = matrix[:-1] + r'\\'
    if array_name != None:
        display(Math(array_name+r' = \begin{bmatrix}'+matrix+r'\end{bmatrix}'))
    else:
        display(Math(r'\begin{bmatrix}'+matrix+r'\end{bmatrix}'))

In [ ]:
%matplotlib widget

import numpy as np
from scipy.integrate import quad
from scipy.linalg import expm
from qutip import *
import numba
from numba import njit, prange
import time
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import sys
import os

In [ ]:
sz = np.array(([[1.0,0.0], [0.0,-1.0]]), dtype=complex); sx = np.array(([[0.0,1.0],[1.0,0.0]]), dtype=complex); sy = np.array(([[0.0,-1j],[1j,0.0]]), dtype=complex) ; sm = np.array(([[0.0, 1.0],[0.0,0.0]]), dtype=complex) ; sp = np.array(([[0.0,0.0],[1.0,0.0]]), dtype=complex)

# ==========================
# Input Parsing from Bash
# ==========================
theta_deg = 0.0  # Default fallback if run manually

# --- Must match the values used in the main simulation script ---
dt = 1.0
N_traj = 10000

dt_str = f"{dt:.2f}".replace(".", "p")
theta_str = f"{theta_deg:.3f}".replace(".", "p")

results_dir = "../Results/Data/"
Output_dir = f"../Results/Plot/Populations/{theta_str}"
os.makedirs(Output_dir, exist_ok=True)

fname = os.path.join(results_dir, f"result_FMO_theta{theta_str}_dt{dt_str}_Ntraj{N_traj}.npz")

try:
    data = np.load(fname)
    print(f"Data extraction completed successfully for Theta = {theta_deg} deg")
except FileNotFoundError:
    print(f"Error: File {fname} not found. Ensure the simulation for this angle has completed.")
    sys.exit(1)

times = data['times']
dt_val = float(data['dt'])
N_site = int(data['N_site'])
eigenergies = data['eigenergies']
eigenvectors = data['eigenvectors']
psi0_exc = data['psi0_exc']

total_jumps = data['total_jumps']
jump_counts = data['jump_counts']            # (n_times, n_traj)

psi_traj_exc = data['psi_traj']                  # (N_site, n_times, n_traj), exciton basis, complex64
pop_traj_avg = data['pop_traj_mean']        # (n_times, N_site)
pop_traj_stderr = data['pop_traj_stderr']

rho_redfield_site = data['rho_redfield_site']       # (n_times, N_site, N_site)
rho_trace_coll_site = data['rho_trace_coll_site']  # (n_times, N_site, N_site)
rho_traj_avg_site = data['rho_traj_avg_site']        # (n_times, N_site, N_site)

n_times, n_traj = jump_counts.shape

# ==========================
# Site-basis single-trajectory populations
# psi_traj is in the exciton basis -> transform to site basis
# ==========================
psi_traj_site = np.einsum('ia,atk->itk', eigenvectors, psi_traj_exc)   # (N_site, n_times, n_traj)
pop_traj_site = np.abs(psi_traj_site) ** 2                          # (N_site, n_times, n_traj)

# ==========================
# Isolated system (no collisions): recomputed on the fly from
# eigenergies, eigenvectors, psi0_exc, times -- no need to store it upfront
# ==========================
phase = np.exp(-1j * np.outer(times, eigenergies))          # (n_times, N)
psi_iso_exc = phase * psi0_exc[None, :]                      # (n_times, N)
psi_iso_site = psi_iso_exc @ eigenvectors.T                  # (n_times, N_site)
pop_iso_site = np.abs(psi_iso_site) ** 2                     # (n_times, N_site)

# ==========================
# Redfield / collisional / MC-avg populations (site basis)
# ==========================
pop_redfield_site = np.real(np.diagonal(rho_redfield_site, axis1=1, axis2=2))       # (n_times, N_site)
pop_trace_coll_site = np.real(np.diagonal(rho_trace_coll_site, axis1=1, axis2=2))  # (n_times, N_site)
pop_traj_avg_site = np.real(np.diagonal(rho_traj_avg_site, axis1=1, axis2=2))              # should ~ pop_traj_avg


# ===========================
# General plot setup
# ===========================
plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 9,
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': ':',
    'figure.autolayout': True
})

def save_fig(fig, filename):
    path_png = os.path.join(Output_dir, f"{filename}.png")
    fig.savefig(path_png, dpi=300, bbox_inches='tight')
    print(f"Saved: {path_png}")
    plt.close(fig)

SITE_LABELS = [f"Site {i+1}" for i in range(N_site)]


In [ ]:
# ==========================
# Identify trajectories that experienced at least one jump (theta=0 only meaningful)
# Uses the exact jump record, not a population-threshold heuristic
# ==========================
n_jumps_per_traj = jump_counts.sum(axis=0)   # (n_traj,)
jump_indices = np.where(n_jumps_per_traj > 0)[0]
print(f"Total trajectories: {n_traj}")
print(f"Trajectories with at least one jump: {len(jump_indices)}")
sample_idx = jump_indices[0] if len(jump_indices) > 0 else 0
print(f"Selected sample_idx for single-trajectory plots: {sample_idx}")


sites_to_plot = [0, 1, 2, 3, 4, 5, 6]   # sites 1, 2, 3

fig2, axes2 = plt.subplots(len(sites_to_plot), 1, figsize=(8, 4 * len(sites_to_plot)))

# Loop through the axes and sites to plot the data
for ax, i in zip(np.atleast_1d(axes2), sites_to_plot):
    ax.plot(times, pop_traj_site[i, :, sample_idx], label='Single trajectory', linewidth=1.8, color='blue', alpha=0.85)
    ax.plot(times, pop_iso_site[:, i], label='Isolated system (no collisions)', linewidth=2, linestyle=':', color='red')
    ax.plot(times, pop_redfield_site[:, i], label='Redfield', linewidth=1.5, linestyle='--', color='black', alpha=0.8)
    ax.set_title(SITE_LABELS[i])
    ax.set_xlabel('Time (fs)')
    ax.set_ylabel('Population')
    ax.legend(loc='best')

# Add the main title to the figure
# The parameter y=1.02 shifts the title slightly upwards to prevent overlap
fig2.suptitle(f'Theta={theta_deg} deg - Single Trajectory (idx={sample_idx}) vs Isolated System', y=1.02)

# Adjust layout to prevent overlap between the vertically stacked subplots
fig2.tight_layout()

# Display the interactive figure
plt.show()